# Diagnosis graph (test)

In [27]:
##########################################################################
### COPY THIS CODE AT THE TOP OF EVERY NOTEBOOK ##########################
##########################################################################

# File navigation imports
import sys
from pathlib import Path
DATA_LOADING_DIRECTORY = (Path("../Preprocessing/").resolve())
sys.path.append(str(DATA_LOADING_DIRECTORY))

# Preprocessing/data_loading.py
from data_loading import *

# Preprocessing/encode.ipynb
from encode import *

# FeatureEngineering/diagnosis_graph.py
from diagnosis_graph import DiagnosisGraph

##########################################################################
##########################################################################
##########################################################################

In [28]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [29]:
# example config 
FILENAME = 'diabetes_clean' 
FOLDER = 'Cleaned'
df_clean = read_data(FILENAME, FOLDER) # cleaned data

In [30]:
DIAGNOSIS_COLUMNS = ['diag_1', 'diag_2', 'diag']

In [33]:
import itertools
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------
# Assumptions
# ---------------------------------------------------------------------
# - df_clean already exists.
# - DiagnosisGraph class already exists.
# - diag_1, diag_2, diag_3 have already been normalised to non-decimal stems.
# - Missing / NaN values have already been handled.
# - Target column is already binary 0/1: readmitted_30_days.

TARGET_COL = "readmitted_30_days"
DIAGNOSTIC_COLUMNS = ("diag_1", "diag_2", "diag_3")
UNKNOWN = globals().get("UNKNOWN", "Unknown")

ID_COLUMNS = [
    "encounter_id",
    "patient_nbr",
]

LEAKAGE_COLUMNS = [
    "readmitted_30_days",
    "readmitted",
]

OUTER_K = 10
INNER_K = 3
RANDOM_STATE = 42

GRAPH_PARAMS = {
    "diag_cols": DIAGNOSTIC_COLUMNS,
    "unknown_token": UNKNOWN,
    "use_ppmi": True,
    "min_pair_count": 5,
    "weighted_edges": False,
    "weighted_nodes": False,
}

LOGREG_PARAM_GRID = [
    {"C": C, "class_weight": class_weight}
    for C, class_weight in itertools.product(
        [0.01, 0.1, 1.0, 10.0],
        [None, "balanced"],
    )
]

# Numeric columns that are actually categorical in this dataset.
# These are one-hot encoded if present.
CATEGORICAL_LIKE_COLUMNS = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]


# ---------------------------------------------------------------------
# Target / feature split
# ---------------------------------------------------------------------

if TARGET_COL not in df_clean.columns:
    raise ValueError(f"Expected target column '{TARGET_COL}' not found in df_clean.")

y = df_clean[TARGET_COL].astype(int)

X = df_clean.drop(
    columns=[col for col in LEAKAGE_COLUMNS if col in df_clean.columns],
    errors="ignore",
)

print("X shape:", X.shape)
print("Target distribution:")
print(y.value_counts(dropna=False))
print(y.value_counts(normalize=True, dropna=False).rename("proportion"))

assert y.nunique() == 2, "Target must contain both 0 and 1."


# ---------------------------------------------------------------------
# From-scratch stratified k-fold splitter
# ---------------------------------------------------------------------

def stratified_kfold_indices(y_values, k=10, random_state=42, verbose=False):
    """
    From-scratch stratified k-fold splitter.

    Returns:
        list of (train_idx, valid_idx), where indices are positional.
    """
    y_values = np.asarray(y_values).astype(int)
    rng = np.random.default_rng(random_state)

    classes, class_counts = np.unique(y_values, return_counts=True)

    if len(classes) < 2:
        raise ValueError(f"Cannot split because y has only one class: {classes}")

    if class_counts.min() < k:
        raise ValueError(
            f"Cannot create {k} folds because the minority class has only "
            f"{class_counts.min()} samples."
        )

    folds = [[] for _ in range(k)]

    for cls in classes:
        cls_idx = np.where(y_values == cls)[0]
        rng.shuffle(cls_idx)

        cls_splits = np.array_split(cls_idx, k)

        for fold_id in range(k):
            folds[fold_id].extend(cls_splits[fold_id].tolist())

    all_idx = np.arange(len(y_values))
    output = []

    for fold_id in range(k):
        valid_idx = np.array(folds[fold_id], dtype=int)
        rng.shuffle(valid_idx)

        train_idx = np.setdiff1d(all_idx, valid_idx, assume_unique=False)

        if len(np.unique(y_values[train_idx])) < 2:
            raise ValueError(f"Training split in fold {fold_id} has only one class.")

        if len(np.unique(y_values[valid_idx])) < 2:
            raise ValueError(f"Validation split in fold {fold_id} has only one class.")

        if verbose:
            print(
                f"Fold {fold_id + 1}: "
                f"train positive rate={y_values[train_idx].mean():.4f}, "
                f"valid positive rate={y_values[valid_idx].mean():.4f}, "
                f"train n={len(train_idx)}, valid n={len(valid_idx)}"
            )

        output.append((train_idx, valid_idx))

    return output


# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------

def binary_classification_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "positive_rate_true": float(np.mean(y_true)),
        "positive_rate_pred": float(np.mean(y_pred)),
        "mean_predicted_probability": float(np.mean(y_prob)),
    }


# ---------------------------------------------------------------------
# Fold-safe feature builder
# ---------------------------------------------------------------------

def make_fold_features_for_logreg(
    X_train_fold,
    X_valid_fold,
    graph_params=GRAPH_PARAMS,
    diagnostic_cols=DIAGNOSTIC_COLUMNS,
    id_cols=ID_COLUMNS,
    categorical_like_cols=CATEGORICAL_LIKE_COLUMNS,
    scale_features=True,
):
    """
    Fold-safe preprocessing.

    Fits on X_train_fold only:
        - DiagnosisGraph
        - one-hot encoded column set
        - numeric imputation medians
        - scaling means/stds

    Applies fitted preprocessing to X_valid_fold.
    """

    # 1. Diagnosis graph fitted on training fold only.
    graph = DiagnosisGraph(**graph_params)

    X_train_graph, X_valid_graph = graph.fit_transform_fold(
        X_train_fold=X_train_fold,
        X_valid_fold=X_valid_fold,
    )

    # 2. Base table for ordinary tabular features.
    drop_cols = list(diagnostic_cols) + list(id_cols)

    X_train_base = X_train_fold.drop(columns=drop_cols, errors="ignore").copy()
    X_valid_base = X_valid_fold.drop(columns=drop_cols, errors="ignore").copy()

    # Object/category/bool columns plus known numeric categorical ID columns.
    categorical_cols = X_train_base.select_dtypes(
        include=["object", "category", "bool"],
    ).columns.tolist()

    categorical_cols += [
        col for col in categorical_like_cols
        if col in X_train_base.columns and col not in categorical_cols
    ]

    categorical_cols = sorted(set(categorical_cols))

    for col in categorical_cols:
        X_train_base[col] = X_train_base[col].astype("object").where(
            X_train_base[col].notna(),
            UNKNOWN,
        )
        X_valid_base[col] = X_valid_base[col].astype("object").where(
            X_valid_base[col].notna(),
            UNKNOWN,
        )

    X_train_ohe = pd.get_dummies(
        X_train_base,
        columns=categorical_cols,
        dummy_na=False,
        drop_first=False,
        dtype=np.float32,
    )

    X_valid_ohe = pd.get_dummies(
        X_valid_base,
        columns=categorical_cols,
        dummy_na=False,
        drop_first=False,
        dtype=np.float32,
    )

    # Validation columns are aligned to training columns only.
    X_valid_ohe = X_valid_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

    # 3. Concatenate ordinary features and graph features.
    X_train_final = pd.concat(
        [
            X_train_ohe.reset_index(drop=True),
            X_train_graph.reset_index(drop=True),
        ],
        axis=1,
    )

    X_valid_final = pd.concat(
        [
            X_valid_ohe.reset_index(drop=True),
            X_valid_graph.reset_index(drop=True),
        ],
        axis=1,
    )

    X_valid_final = X_valid_final.reindex(columns=X_train_final.columns, fill_value=0)

    # 4. Numeric conversion and imputation fitted on training fold only.
    X_train_final = X_train_final.apply(pd.to_numeric, errors="coerce")
    X_valid_final = X_valid_final.apply(pd.to_numeric, errors="coerce")

    train_medians = X_train_final.median(axis=0, skipna=True).fillna(0.0)

    X_train_final = X_train_final.fillna(train_medians)
    X_valid_final = X_valid_final.fillna(train_medians)

    # 5. Scaling fitted on training fold only.
    if scale_features:
        train_means = X_train_final.mean(axis=0)
        train_stds = X_train_final.std(axis=0, ddof=0).replace(0, 1.0).fillna(1.0)

        X_train_final = (X_train_final - train_means) / train_stds
        X_valid_final = (X_valid_final - train_means) / train_stds

    X_train_final = X_train_final.astype(np.float32)
    X_valid_final = X_valid_final.astype(np.float32)

    assert list(X_train_final.columns) == list(X_valid_final.columns)
    assert np.isfinite(X_train_final.to_numpy()).all()
    assert np.isfinite(X_valid_final.to_numpy()).all()

    return X_train_final, X_valid_final, graph


# ---------------------------------------------------------------------
# Model builder
# ---------------------------------------------------------------------

def build_logistic_regression(params):
    return LogisticRegression(
        penalty="l2",
        C=params["C"],
        class_weight=params["class_weight"],
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_STATE,
    )


# ---------------------------------------------------------------------
# Nested CV
# ---------------------------------------------------------------------

outer_folds = stratified_kfold_indices(
    y_values=y.to_numpy(),
    k=OUTER_K,
    random_state=RANDOM_STATE,
    verbose=True,
)

outer_results = []
inner_results = []
outer_predictions = []

for outer_fold_id, (outer_train_idx, outer_test_idx) in enumerate(outer_folds, start=1):
    print("\n" + "=" * 60)
    print(f"Outer fold {outer_fold_id}/{OUTER_K}")
    print("=" * 60)

    X_outer_train = X.iloc[outer_train_idx].reset_index(drop=True)
    y_outer_train = y.iloc[outer_train_idx].reset_index(drop=True)

    X_outer_test = X.iloc[outer_test_idx].reset_index(drop=True)
    y_outer_test = y.iloc[outer_test_idx].reset_index(drop=True)

    print("Outer train:", X_outer_train.shape, "positive rate:", y_outer_train.mean())
    print("Outer test: ", X_outer_test.shape, "positive rate:", y_outer_test.mean())

    inner_folds = stratified_kfold_indices(
        y_values=y_outer_train.to_numpy(),
        k=INNER_K,
        random_state=RANDOM_STATE + outer_fold_id,
        verbose=False,
    )

    best_params = None
    best_inner_score = -np.inf

    # -----------------------------
    # Inner CV hyperparameter tuning
    # -----------------------------

    for param_id, params in enumerate(LOGREG_PARAM_GRID, start=1):
        fold_scores = []

        print(f"\n  Params {param_id}/{len(LOGREG_PARAM_GRID)}: {params}")

        for inner_fold_id, (inner_train_idx, inner_valid_idx) in enumerate(inner_folds, start=1):
            X_inner_train = X_outer_train.iloc[inner_train_idx].reset_index(drop=True)
            y_inner_train = y_outer_train.iloc[inner_train_idx].reset_index(drop=True)

            X_inner_valid = X_outer_train.iloc[inner_valid_idx].reset_index(drop=True)
            y_inner_valid = y_outer_train.iloc[inner_valid_idx].reset_index(drop=True)

            X_inner_train_final, X_inner_valid_final, _ = make_fold_features_for_logreg(
                X_train_fold=X_inner_train,
                X_valid_fold=X_inner_valid,
                graph_params=GRAPH_PARAMS,
                diagnostic_cols=DIAGNOSTIC_COLUMNS,
                id_cols=ID_COLUMNS,
                categorical_like_cols=CATEGORICAL_LIKE_COLUMNS,
                scale_features=True,
            )

            model = build_logistic_regression(params)
            model.fit(X_inner_train_final, y_inner_train)

            y_inner_prob = model.predict_proba(X_inner_valid_final)[:, 1]

            metrics = binary_classification_metrics(
                y_true=y_inner_valid,
                y_prob=y_inner_prob,
                threshold=0.5,
            )

            metrics.update({
                "outer_fold": outer_fold_id,
                "inner_fold": inner_fold_id,
                "param_id": param_id,
                "C": params["C"],
                "class_weight": str(params["class_weight"]),
            })

            inner_results.append(metrics)
            fold_scores.append(metrics["roc_auc"])

            print(
                f"    Inner fold {inner_fold_id}/{INNER_K}: "
                f"ROC-AUC={metrics['roc_auc']:.4f}, "
                f"AP={metrics['average_precision']:.4f}"
            )

        mean_inner_score = float(np.mean(fold_scores))
        print(f"  Mean inner ROC-AUC: {mean_inner_score:.4f}")

        if mean_inner_score > best_inner_score:
            best_inner_score = mean_inner_score
            best_params = params.copy()

    print(f"\nBest params: {best_params}")
    print(f"Best mean inner ROC-AUC: {best_inner_score:.4f}")

    # -----------------------------
    # Final fit on full outer train, evaluate once on outer test
    # -----------------------------

    X_outer_train_final, X_outer_test_final, final_graph = make_fold_features_for_logreg(
        X_train_fold=X_outer_train,
        X_valid_fold=X_outer_test,
        graph_params=GRAPH_PARAMS,
        diagnostic_cols=DIAGNOSTIC_COLUMNS,
        id_cols=ID_COLUMNS,
        categorical_like_cols=CATEGORICAL_LIKE_COLUMNS,
        scale_features=True,
    )

    final_model = build_logistic_regression(best_params)
    final_model.fit(X_outer_train_final, y_outer_train)

    y_outer_prob = final_model.predict_proba(X_outer_test_final)[:, 1]

    outer_metrics = binary_classification_metrics(
        y_true=y_outer_test,
        y_prob=y_outer_prob,
        threshold=0.5,
    )

    graph_summary = final_graph.graph_summary()

    outer_metrics.update({
        "outer_fold": outer_fold_id,
        "best_C": best_params["C"],
        "best_class_weight": str(best_params["class_weight"]),
        "best_mean_inner_roc_auc": best_inner_score,
        "n_outer_train": len(X_outer_train),
        "n_outer_test": len(X_outer_test),
        "n_features": X_outer_train_final.shape[1],
        "graph_num_nodes": graph_summary["num_nodes"],
        "graph_num_raw_edges": graph_summary["num_raw_edges"],
        "graph_num_retained_edges": graph_summary["num_retained_edges"],
    })

    outer_results.append(outer_metrics)

    outer_predictions.append(
        pd.DataFrame({
            "outer_fold": outer_fold_id,
            "row_position": outer_test_idx,
            "y_true": y_outer_test.to_numpy(),
            "y_prob": y_outer_prob,
            "y_pred": (y_outer_prob >= 0.5).astype(int),
        })
    )

    print(
        f"\nOuter fold {outer_fold_id}: "
        f"ROC-AUC={outer_metrics['roc_auc']:.4f}, "
        f"AP={outer_metrics['average_precision']:.4f}, "
        f"F1={outer_metrics['f1']:.4f}, "
        f"Recall={outer_metrics['recall']:.4f}, "
        f"Precision={outer_metrics['precision']:.4f}"
    )

outer_results_df = pd.DataFrame(outer_results)
inner_results_df = pd.DataFrame(inner_results)
outer_predictions_df = pd.concat(outer_predictions, ignore_index=True)

print("\nNested CV complete.")
display(outer_results_df)

X shape: (71518, 47)
Target distribution:
readmitted_30_days
0    65225
1     6293
Name: count, dtype: int64
readmitted_30_days
0    0.912008
1    0.087992
Name: proportion, dtype: float64
Fold 1: train positive rate=0.0880, valid positive rate=0.0881, train n=64365, valid n=7153
Fold 2: train positive rate=0.0880, valid positive rate=0.0881, train n=64365, valid n=7153
Fold 3: train positive rate=0.0880, valid positive rate=0.0881, train n=64365, valid n=7153
Fold 4: train positive rate=0.0880, valid positive rate=0.0879, train n=64366, valid n=7152
Fold 5: train positive rate=0.0880, valid positive rate=0.0879, train n=64366, valid n=7152
Fold 6: train positive rate=0.0880, valid positive rate=0.0880, train n=64367, valid n=7151
Fold 7: train positive rate=0.0880, valid positive rate=0.0880, train n=64367, valid n=7151
Fold 8: train positive rate=0.0880, valid positive rate=0.0880, train n=64367, valid n=7151
Fold 9: train positive rate=0.0880, valid positive rate=0.0880, train n=643

C:\Users\RICT\AppData\Local\Temp\ipykernel_42440\2529172394.py:235: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train_base.select_dtypes(


    Inner fold 1/3: ROC-AUC=0.6508, AP=0.1654


C:\Users\RICT\AppData\Local\Temp\ipykernel_42440\2529172394.py:235: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train_base.select_dtypes(


    Inner fold 2/3: ROC-AUC=0.6507, AP=0.1660


...

C:\Users\RICT\AppData\Local\Temp\ipykernel_42440\2529172394.py:235: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train_base.select_dtypes(


    Inner fold 3/3: ROC-AUC=0.6492, AP=0.1645
  Mean inner ROC-AUC: 0.6474

Best params: {'C': 0.01, 'class_weight': 'balanced'}
Best mean inner ROC-AUC: 0.6483


C:\Users\RICT\AppData\Local\Temp\ipykernel_42440\2529172394.py:235: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train_base.select_dtypes(



Outer fold 10: ROC-AUC=0.6598, AP=0.1695, F1=0.2229, Recall=0.5628, Precision=0.1389

Nested CV complete.


,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,specificity,f1,tp,fp,tn,fn,positive_rate_true,positive_rate_pred,mean_predicted_probability,outer_fold,best_C,best_class_weight,best_mean_inner_roc_auc,n_outer_train,n_outer_test,n_features,graph_num_nodes,graph_num_raw_edges,graph_num_retained_edges
0,0.669887,0.178907,0.657347,0.622839,0.143361,0.580952,0.664725,0.229972,366,2187,4336,264,0.088075,0.356913,0.460295,1,0.01,balanced,0.651588,64365,7153,376,21,186,155
1,0.650446,0.170365,0.647560,0.604567,0.134519,0.552381,0.656753,0.216351,348,2239,4284,282,0.088075,0.361666,0.459999,2,0.01,balanced,0.650883,64365,7153,375,21,184,158
2,0.650959,0.162213,0.643227,0.597889,0.131236,0.542857,0.652920,0.211372,342,2264,4259,288,0.088075,0.364323,0.459437,3,0.01,balanced,0.653222,64365,7153,376,21,186,160
3,0.650826,0.166745,0.651566,0.612180,0.137971,0.564388,0.659972,0.221736,355,2218,4305,274,0.087947,0.359760,0.458705,4,0.01,balanced,0.653070,64366,7152,377,21,186,159
4,0.655514,0.163777,0.643037,0.599604,0.131700,0.546900,0.652307,0.212280,344,2268,4255,285,0.087947,0.365213,0.459418,5,0.01,balanced,0.648039,64366,7152,375,21,186,158
5,0.654065,0.160737,0.661726,0.614159,0.140562,0.556439,0.671880,0.224431,350,2140,4382,279,0.087960,0.348203,0.454309,6,0.01,balanced,0.653296,64367,7151,373,21,185,159
6,0.657931,0.165540,0.653755,0.609789,0.137417,0.556439,0.663140,0.220403,350,2197,4325,279,0.087960,0.356174,0.458379,7,0.01,balanced,0.651711,64367,7151,376,21,186,160
7,0.648148,0.173829,0.644246,0.604576,0.133843,0.556439,0.652714,0.215783,350,2265,4257,279,0.087960,0.365683,0.460999,8,0.01,balanced,0.655450,64367,7151,374,21,185,158
8,0.654288,0.174677,0.651937,0.603047,0.134434,0.543720,0.662374,0.215569,342,2202,4320,287,0.087960,0.355754,0.459174,9,0.01,balanced,0.649831,64367,7151,375,21,186,159
9,0.659761,0.169482,0.654734,0.613199,0.138932,0.562798,0.663600,0.222852,354,2194,4328,275,0.087960,0.356314,0.459010,10,0.01,balanced,0.648253,64367,7151,374,21,186,157


In [34]:
METRIC_COLUMNS = [
    "roc_auc",
    "average_precision",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "positive_rate_true",
    "positive_rate_pred",
    "mean_predicted_probability",
]

summary_rows = []

for metric in METRIC_COLUMNS:
    values = outer_results_df[metric].astype(float).to_numpy()

    n = np.sum(~np.isnan(values))
    mean = np.nanmean(values)
    sd = np.nanstd(values, ddof=1)
    se = sd / np.sqrt(n)
    ci95_low = mean - 1.96 * se
    ci95_high = mean + 1.96 * se

    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "sd": sd,
        "se": se,
        "ci95_low": ci95_low,
        "ci95_high": ci95_high,
    })

metrics_summary_df = pd.DataFrame(summary_rows)

print("Outer-fold performance summary:")
display(metrics_summary_df)

print("Selected hyperparameters and outer-fold scores:")
display(
    outer_results_df[
        [
            "outer_fold",
            "best_C",
            "best_class_weight",
            "best_mean_inner_roc_auc",
            "roc_auc",
            "average_precision",
            "balanced_accuracy",
            "f1",
            "recall",
            "precision",
            "specificity",
            "n_features",
            "graph_num_retained_edges",
        ]
    ]
)

print("Confusion matrix totals across all outer folds:")

total_tp = int(outer_results_df["tp"].sum())
total_fp = int(outer_results_df["fp"].sum())
total_tn = int(outer_results_df["tn"].sum())
total_fn = int(outer_results_df["fn"].sum())

confusion_totals = pd.DataFrame(
    {
        "Predicted 0": [total_tn, total_fn],
        "Predicted 1": [total_fp, total_tp],
    },
    index=["Actual 0", "Actual 1"],
)

display(confusion_totals)

print("Pooled out-of-fold metrics:")

pooled_metrics = binary_classification_metrics(
    y_true=outer_predictions_df["y_true"],
    y_prob=outer_predictions_df["y_prob"],
    threshold=0.5,
)

pooled_metrics_df = pd.DataFrame([pooled_metrics])
display(pooled_metrics_df)

# Optional: inspect tuning behaviour across inner folds.
print("Mean inner-CV ROC-AUC by hyperparameter setting:")
inner_param_summary = (
    inner_results_df
    .groupby(["C", "class_weight"], as_index=False)
    .agg(
        mean_inner_roc_auc=("roc_auc", "mean"),
        sd_inner_roc_auc=("roc_auc", "std"),
        mean_inner_ap=("average_precision", "mean"),
        n=("roc_auc", "count"),
    )
    .sort_values("mean_inner_roc_auc", ascending=False)
)

display(inner_param_summary)

Outer-fold performance summary:


,metric,mean,sd,se,ci95_low,ci95_high
0,roc_auc,0.655182,0.006286,0.001988,0.651286,0.659079
1,average_precision,0.168627,0.005898,0.001865,0.164972,0.172283
2,accuracy,0.650913,0.006318,0.001998,0.646997,0.654829
3,balanced_accuracy,0.608185,0.007647,0.002418,0.603445,0.612924
4,precision,0.136397,0.003921,0.001240,0.133967,0.138828
5,recall,0.556331,0.011344,0.003587,0.549300,0.563362
6,specificity,0.660039,0.006367,0.002013,0.656092,0.663985
7,f1,0.219075,0.005837,0.001846,0.215457,0.222692
8,positive_rate_true,0.087992,0.000058,0.000018,0.087956,0.088028
9,positive_rate_pred,0.359000,0.005436,0.001719,0.355631,0.362369


Selected hyperparameters and outer-fold scores:


,outer_fold,best_C,best_class_weight,best_mean_inner_roc_auc,roc_auc,average_precision,balanced_accuracy,f1,recall,precision,specificity,n_features,graph_num_retained_edges
0,1,0.01,balanced,0.651588,0.669887,0.178907,0.622839,0.229972,0.580952,0.143361,0.664725,376,155
1,2,0.01,balanced,0.650883,0.650446,0.170365,0.604567,0.216351,0.552381,0.134519,0.656753,375,158
2,3,0.01,balanced,0.653222,0.650959,0.162213,0.597889,0.211372,0.542857,0.131236,0.652920,376,160
3,4,0.01,balanced,0.653070,0.650826,0.166745,0.612180,0.221736,0.564388,0.137971,0.659972,377,159
4,5,0.01,balanced,0.648039,0.655514,0.163777,0.599604,0.212280,0.546900,0.131700,0.652307,375,158
5,6,0.01,balanced,0.653296,0.654065,0.160737,0.614159,0.224431,0.556439,0.140562,0.671880,373,159
6,7,0.01,balanced,0.651711,0.657931,0.165540,0.609789,0.220403,0.556439,0.137417,0.663140,376,160
7,8,0.01,balanced,0.655450,0.648148,0.173829,0.604576,0.215783,0.556439,0.133843,0.652714,374,158
8,9,0.01,balanced,0.649831,0.654288,0.174677,0.603047,0.215569,0.543720,0.134434,0.662374,375,159
9,10,0.01,balanced,0.648253,0.659761,0.169482,0.613199,0.222852,0.562798,0.138932,0.663600,374,157


Confusion matrix totals across all outer folds:


,Predicted 0,Predicted 1
Actual 0,43051,22174
Actual 1,2792,3501


Pooled out-of-fold metrics:


,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,specificity,f1,tp,fp,tn,fn,positive_rate_true,positive_rate_pred,mean_predicted_probability
0,0.655122,0.166927,0.650913,0.608185,0.136358,0.556332,0.660038,0.219032,3501,22174,43051,2792,0.087992,0.359001,0.458973


Mean inner-CV ROC-AUC by hyperparameter setting:


,C,class_weight,mean_inner_roc_auc,sd_inner_roc_auc,mean_inner_ap,n
1,0.01,balanced,0.651534,0.005375,0.165177,30
3,0.10,balanced,0.651035,0.005359,0.164931,30
5,1.00,balanced,0.650887,0.005358,0.164751,30
7,10.00,balanced,0.650809,0.005382,0.164707,30
0,0.01,None,0.650549,0.005822,0.167114,30
2,0.10,None,0.649870,0.005857,0.166836,30
4,1.00,None,0.649599,0.005784,0.166669,30
6,10.00,None,0.649498,0.005814,0.166621,30
